# Kaggle GPU smoke test — FL-for-Aircraft

Runs on **Kaggle's Jupyter Server** (connected from VS Code). Validates the
whole pipeline before any real GPU campaign:

1. Confirm the kernel sees Kaggle's GPU
2. Clone the `multiseed` branch (public, no token)
3. Confirm the CMAPSS data came with the clone
4. Install the one extra dependency (captum)
5. Add `src/` to the path, move the model to GPU, run a forward pass

> Cells execute on **Kaggle**, not your laptop. We do NOT run
> `pip install -e .` because pyproject pins Python 3.12 and Kaggle may run a
> different minor version; adding `src/` to `sys.path` sidesteps that.

## 1. Environment + GPU check

In [1]:
import platform
import torch

print(f"Python      : {platform.python_version()}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA build  : {torch.version.cuda}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count   : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  cuda:{i}    : {torch.cuda.get_device_name(i)}")
else:
    print("WARNING: no GPU visible - set Accelerator to GPU T4 x2 in Notebook Settings")

Python      : 3.12.13
PyTorch     : 2.10.0+cu128
CUDA build  : 12.8
CUDA avail  : True
GPU count   : 2
  cuda:0    : Tesla T4
  cuda:1    : Tesla T4


## 2. Clone the multiseed branch (anonymous - no token)

In [2]:
import os

REPO_DIR = "/kaggle/working/FL-for-Aircraft"
BRANCH = "multiseed"
if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} --depth 1 https://github.com/Chinmoy17/FL-for-Aircraft.git {REPO_DIR}
else:
    print("Repo already cloned; pulling latest")
    !cd {REPO_DIR} && git pull --ff-only

%cd {REPO_DIR}
!git log --oneline -1

Repo already cloned; pulling latest
Already up to date.
/kaggle/working/FL-for-Aircraft
2b1dddc (grafted, HEAD -> main, origin/main, origin/HEAD) Readme Updated


## 3. Confirm CMAPSS data came with the clone

In [3]:
!ls -la Dataset/CMAPSS_NASA/ | head -12

total 44332
drwxr-xr-x 2 root root     4096 Jul 27 06:29 .
drwxr-xr-x 3 root root     4096 Jul 27 06:29 ..
-rw-r--r-- 1 root root   434158 Jul 27 06:29 Damage Propagation Modeling.pdf
-rw-r--r-- 1 root root     2442 Jul 27 06:29 readme.txt
-rw-r--r-- 1 root root      429 Jul 27 06:29 RUL_FD001.txt
-rw-r--r-- 1 root root     1110 Jul 27 06:29 RUL_FD002.txt
-rw-r--r-- 1 root root      428 Jul 27 06:29 RUL_FD003.txt
-rw-r--r-- 1 root root     1084 Jul 27 06:29 RUL_FD004.txt
-rw-r--r-- 1 root root  2228855 Jul 27 06:29 test_FD001.txt
-rw-r--r-- 1 root root  5734587 Jul 27 06:29 test_FD002.txt
-rw-r--r-- 1 root root  2826651 Jul 27 06:29 test_FD003.txt


## 4. Install the one extra dependency

Kaggle already ships torch / numpy / pandas / scikit-learn / pyyaml / tqdm.
Only `captum` (used by the interpretability module) is extra. We skip
`pip install -e .` on purpose (Python-version pin).

In [4]:
!pip install captum -q
print("deps done")

deps done


## 5. GPU forward-pass smoke test

Adds `src/` to the path, builds the multi-task CNN, moves it plus a dummy
batch to the GPU, and runs a forward pass. Proves the model is
GPU-compatible on Kaggle even before we wire device support into the
training loop.

In [5]:
import sys

SRC = "/kaggle/working/FL-for-Aircraft/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import torch
from fl_aircraft.models import MultiTaskCNN, MultiTaskCNNConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MultiTaskCNN(MultiTaskCNNConfig(n_features=17, window_size=30)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"model params : {n_params:,}")
print(f"device       : {device}")

x = torch.randn(8, 30, 17, device=device)
model.eval()
with torch.no_grad():
    pred = model(x)
print(f"forward OK   : rul={tuple(pred.rul.shape)}, on {pred.rul.device}")
print("\nSMOKE TEST PASSED" if str(pred.rul.device).startswith(device) else "device mismatch")

model params : 30,018
device       : cuda
forward OK   : rul=(8,), on cuda:0

SMOKE TEST PASSED


In [ ]:
print("All done. You can now run the notebook cells below to train and evaluate the model on the CMAPSS dataset.")